[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/02_text_chunking/02_text_chunking.ipynb)

# 02. 텍스트 청킹 & PDF 처리 실습 (langchain-text-splitters + PyMuPDF/pypdf + tiktoken)

> 관련 예제 프로젝트: [`example-projects/preprocess-example`](../../../example-projects/preprocess-example) (A파트-2: 전처리), [`example-projects/rag-regulation-example`](../../../example-projects/rag-regulation-example) 의 `ingest.py`

[01_web_crawling.ipynb](../01_web_crawling/01_web_crawling.ipynb)에서 모아둔 원본 텍스트나 PDF는 "사람이 읽는 형태"일 뿐, AI 검색에 바로 쓸 수 없습니다. 이 노트북에서는 문서를 작은 조각(청크)으로 자르고, PDF에서 글자를 뽑아내고, 토큰 수를 세는 방법을 실습합니다.

## 이론

### 1) 왜 청킹이 필요한가?
- AI 모델은 한 번에 읽을 수 있는 글자(토큰) 양에 한계가 있습니다.
- 문서 전체보다 작은 단위로 나눠야 질문과 "정확히 관련된 부분만" 찾아낼 수 있어 검색 정확도가 높아집니다.

### 2) `RecursiveCharacterTextSplitter`의 동작 방식
`separators=["\n\n", "\n", ". ", " ", ""]` 순서대로, 먼저 문단 구분으로 잘라보고 그래도 `chunk_size`보다 길면 줄바꿈 -> 마침표 -> 공백 -> 글자 단위로 점점 더 잘게 잘라서 크기를 맞춥니다. `chunk_overlap`은 조각 경계에서 문맥이 끊기지 않도록 옆 조각과 겹치는 글자 수입니다.

### 3) PDF에서 글자 뽑기: PyMuPDF(`fitz`) vs `pypdf`
둘 다 PDF에서 텍스트를 추출하는 라이브러리입니다. `PyMuPDF`는 빠르고 레이아웃 보존이 좋아 `preprocess-example`에서 쓰이고, `pypdf`는 순수 파이썬이라 설치가 가볍고 `rag-regulation-example`의 `PyPDFLoader`가 내부적으로 사용합니다.

### 4) 토큰(token)이란?
OpenAI 모델의 과금/컨텍스트 제한은 "글자 수"가 아니라 "토큰 수" 기준입니다. `tiktoken`으로 실제 모델이 텍스트를 몇 개의 토큰으로 쪼개는지 확인할 수 있습니다.

## 실습 0. 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q langchain-text-splitters langchain-core pypdf pymupdf tiktoken fpdf2

## 실습 1. `RecursiveCharacterTextSplitter`로 텍스트 청킹하기

`preprocess.py`와 동일하게 `chunk_size=500`, `chunk_overlap=75`로 사내 규정 예시 텍스트를 잘라봅니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

sample_text = (
    "제1조(목적) 이 규정은 임직원의 휴가 사용에 관한 사항을 정함을 목적으로 한다.\n\n제2조(연차휴가) 1년간 80퍼센트 이상 출근한 임직원에게는 15일의 연차휴가를 부여한다. 3년 이상 계속 근로한 임직원에 대하여는 최초 1년을 초과하는 계속 근로 연수 매 2년에 대하여 1일을 가산한 유급휴가를 부여한다.\n\n제3조(육아휴직) 임신 중인 여성 임직원이나 만 8세 이하 자녀를 양육하는 임직원은 최대 1년의 육아휴직을 신청할 수 있다. 육아휴직 기간은 근속연수에 포함하되, 승급을 위한 근속기간에는 산입하지 아니한다.\n\n제4조(경조휴가) 결혼, 출산, 사망 등 경조사가 발생한 경우 별표에서 정한 일수의 유급휴가를 부여한다. 본인 결혼은 5일, 배우자 출산은 10일, 부모 사망은 5일로 한다.\n\n제5조(병가) 업무 외 질병이나 부상으로 근로가 불가능한 경우 연간 60일 이내에서 병가를 사용할 수 있다. 병가 3일 이상 사용 시 진단서를 제출하여야 한다.\n\n제6조(휴가의 신청) 휴가를 사용하고자 하는 임직원은 사용 예정일 3일 전까지 소속 부서장에게 신청하여야 한다. 다만 병가 등 부득이한 사유가 있는 경우에는 사후에 신청할 수 있다."
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
    separators=["\n\n", "\n", ". ", " ", ""],
)

doc = Document(page_content=sample_text, metadata={"source": "휴가규정.txt"})
chunks = splitter.split_documents([doc])

print(f"{len(chunks)}개 청크 생성")
for i, c in enumerate(chunks):
    print(f"--- chunk {i} ({len(c.page_content)}자) ---")
    print(c.page_content[:120].replace(chr(10), ' '), "...")

## 실습 2. `chunk_size`/`chunk_overlap`을 바꿔가며 청크 개수 변화 관찰하기

청크 크기가 작을수록 더 잘게 쪼개져서 조각 수가 늘어납니다. 조항 번호가 촘촘한 규정 문서일수록 작은 `chunk_size`가 유리하다는 것을 `preprocess-example`의 README에서도 설명하고 있습니다.

In [ ]:
configs = [(1000, 150), (500, 75), (200, 30)]

for size, overlap in configs:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    result = sp.split_documents([doc])
    avg_len = sum(len(c.page_content) for c in result) / len(result)
    print(f"chunk_size={size:<5} overlap={overlap:<4} -> {len(result)}개 청크 (평균 {avg_len:.0f}자)")

## 실습 3. 연습용 PDF 만들기

실습을 위해 `fpdf2`로 간단한 규정 문서 PDF를 직접 생성합니다 (외부 사이트의 PDF 파일에 의존하지 않기 위해서입니다). 한글 폰트를 임베딩하는 과정은 복잡하므로, 이 실습에서는 영문 예시 텍스트로 만듭니다 — `fitz`/`pypdf`의 텍스트 추출 자체는 한글 PDF에도 동일하게 동작합니다.

In [ ]:
from fpdf import FPDF, XPos, YPos

articles = [
    "Article 1 (Purpose). This policy defines the rules for employee leave.",
    "Article 2 (Annual Leave). Employees who attended more than 80 percent of working days "
    "in a year are granted 15 days of annual paid leave.",
    "Article 3 (Parental Leave). An employee raising a child under age 8 may request parental "
    "leave of up to one year.",
    "Article 4 (Sick Leave). An employee unable to work due to illness may use up to 60 days "
    "of sick leave per year. A medical certificate is required for 3 or more consecutive days.",
]

pdf = FPDF()
for i, article in enumerate(articles):
    pdf.add_page()
    pdf.set_font("Helvetica", size=14)
    pdf.multi_cell(0, 10, f"Page {i + 1}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("Helvetica", size=11)
    pdf.multi_cell(0, 8, article, new_x=XPos.LMARGIN, new_y=YPos.NEXT)

pdf.output("sample_policy.pdf")
print("sample_policy.pdf 생성 완료")

## 실습 4. `PyMuPDF`(fitz)로 텍스트 추출하기

`preprocess.py`의 `extract_pdf_text()`와 동일한 방식입니다. 메모리에 있는 바이트를 그대로 열 수도 있고, 파일 경로로 바로 열 수도 있습니다.

In [ ]:
import fitz  # PyMuPDF

def extract_pdf_text_fitz(path: str) -> str:
    text_parts = []
    with fitz.open(path) as pdf:
        for page in pdf:
            text_parts.append(page.get_text())
    return "\n".join(text_parts)


fitz_text = extract_pdf_text_fitz("sample_policy.pdf")
print(fitz_text[:400])

## 실습 5. `pypdf`로 페이지별 텍스트/메타데이터 추출하기

`rag-regulation-example`의 `ingest.py`는 LangChain의 `PyPDFLoader`(내부적으로 `pypdf` 사용)로 페이지별 `Document`를 만들고, 각 조각에 `source`(파일명)와 `page`(페이지 번호) 메타데이터를 붙여둡니다. 여기서는 `pypdf`를 직접 사용해 같은 결과를 만들어봅니다.

In [ ]:
from pypdf import PdfReader

reader = PdfReader("sample_policy.pdf")
page_docs = []
for page_number, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    page_docs.append(
        Document(page_content=text, metadata={"source": "sample_policy.pdf", "page": page_number})
    )

print(f"{len(page_docs)}페이지 로드")
print(page_docs[1].page_content)
print(page_docs[1].metadata)

## 실습 6. PDF 페이지를 청크로 나누고 출처 메타데이터 유지하기

`ingest.py`의 `load_and_split()`처럼, 페이지 단위 `Document` 목록을 통째로 `split_documents()`에 넘기면 각 청크가 원래 페이지의 메타데이터(`source`, `page`)를 그대로 물려받습니다.

In [ ]:
pdf_splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
pdf_chunks = pdf_splitter.split_documents(page_docs)

print(f"{len(pdf_chunks)}개 청크")
for c in pdf_chunks[:3]:
    print(c.metadata, "->", c.page_content[:60])

## 실습 7. `tiktoken`으로 토큰 수 세어보기

`gpt-4o-mini` 계열 모델과 호환되는 `cl100k_base` 인코딩으로, 같은 문장이라도 한글이 영어보다 토큰을 더 많이 소비하는 경향을 확인해봅니다.

In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

samples = {
    "영문": "Employees are granted 15 days of annual paid leave.",
    "한글": "임직원에게는 15일의 연차휴가를 부여한다.",
}

for label, text in samples.items():
    tokens = encoding.encode(text)
    print(f"{label}: 글자 수={len(text):>3}  토큰 수={len(tokens):>3}")

## 정리 & 연습 문제
- **연습 1**: `separators`에 `"\n\n제"`처럼 조항 구분을 우선순위로 넣어서 `sample_text`를 다시 청킹하고, 기본 `separators` 결과와 비교해보세요. `chunk_size`를 여러 값으로 바꿔가며, 정말로 조항 경계가 항상 지켜지는지도 확인해보세요. (힌트: `separators`는 큰 텍스트를 어떻게 쪼갤지만 정할 뿐, 이미 작아진 조각들을 다시 합칠지는 `chunk_size`만으로 결정됩니다 — 이 힌트가 무슨 뜻인지 직접 확인해보세요.)
- **연습 2**: 실습 1의 `chunks` 리스트 각각에 대해 `tiktoken`으로 토큰 수를 계산해서 평균/최댓값을 구하고, `chunk_size=500`(글자 기준)이 실제로는 토큰 기준으로 몇 개 정도인지 확인해보세요.

**해설/정답**: [02_text_chunking_solutions.ipynb](02_text_chunking_solutions.ipynb)

## 다음 단계
다음 노트북([03_document_structuring](../03_document_structuring/03_document_structuring.ipynb))에서는 OCR로 읽어낸 지저분한 텍스트를 Pydantic + OpenAI로 정형화하는 방법을 실습합니다.